Author: Will Blanton

# Imports

In [ ]:
import ast
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import itertools
import sys
from pathlib import Path

from sentence_transformers import SentenceTransformer

from torch.utils.data import Dataset, DataLoader

import optuna
sys.path.insert(0, str(Path("src").resolve()))
from nyt_connections_solver.models import AttentionModel, Baseline, SetTransformer
from nyt_connections_solver.harness import SearchHarness
from nyt_connections_solver.evaluation import load_puzzles
from nyt_connections_solver.game import LocalConnectionsEnvironment, play_game
from nyt_connections_solver.feedback_search import FeedbackSearchHarness
from nyt_connections_solver.evaluation.feedback import evaluate_feedback_games

ModuleNotFoundError: No module named 'nyt_connections_solver.evaluation.feedback'; 'nyt_connections_solver.evaluation' is not a package

# Graph attention experiment
Frozen Qwen text embeddings plus hashed character n-grams and structural spelling
features form the nodes. Shared edge providers compute semantic, English WordNet,
character, prefix/suffix, phrase, anagram, and spelling-structure channels.

Features are cached under `data/feature_cache/graph/`. Full-board tensors are reused
across all partial states. Set `REFRESH_FEATURE_CACHE = True` for a deliberate refresh;
provider versions, options, model revisions, and lexical fingerprints otherwise
invalidate only affected cache entries. Training remains an explicit notebook step.


In [ ]:
from nyt_connections_solver.models import MultiheadAttentionBlock, SetDecoder

class GraphAttentionBlock(MultiheadAttentionBlock):
    def __init__(self, width, heads, dropout, edge_dim, edge_bias_scale):
        super().__init__(width, width, attn_heads=heads, dropout=dropout)
        self.edge_bias_scale = float(edge_bias_scale)
        if self.edge_bias_scale < 0:
            raise ValueError("edge_bias_scale must be nonnegative")
        self.edge_projection = nn.Linear(edge_dim, heads, bias=False)
        # Start from ordinary attention and learn which relationships to use.
        nn.init.zeros_(self.edge_projection.weight)

    def attention_bias(self, edges):
        bias = self.edge_bias_scale * torch.tanh(self.edge_projection(edges))
        # (B, N, N, H) -> (B * H, N, N), as required by MultiheadAttention.
        diagonal = torch.eye(edges.shape[1], device=edges.device, dtype=torch.bool)
        bias = bias.masked_fill(diagonal[None, :, :, None], 0)
        return bias.permute(0, 3, 1, 2).reshape(-1, edges.shape[1], edges.shape[2])

    def forward(self, x, edges):
        # x: (B, N, D); edges: (B, N, N, E)
        bias = None if self.edge_bias_scale == 0 else self.attention_bias(edges).to(x.dtype)
        attended = self.attn(x, x, x, attn_mask=bias, need_weights=False)[0]
        x = self.h_norm(x + attended)
        return self.mab_norm(self.residual_proj(x) + self.ff(x))


class GraphEncoder(nn.Module):
    def __init__(self, width, depth, heads, dropout, edge_dim, edge_bias_scale):
        super().__init__()
        self.mabs = nn.ModuleList([
            GraphAttentionBlock(width, heads, dropout, edge_dim, edge_bias_scale)
            for _ in range(depth)
        ])

    def forward(self, x, edges):
        for block in self.mabs:
            x = block(x, edges)
        return x


class GraphCrossAttentionBlock(MultiheadAttentionBlock):
    def __init__(self, width, heads, dropout, edge_dim, edge_bias_scale):
        super().__init__(width, width, attn_heads=heads, dropout=dropout)
        self.edge_bias_scale = float(edge_bias_scale)
        self.edge_projection = nn.Linear(edge_dim, heads, bias=False)
        nn.init.zeros_(self.edge_projection.weight)

    def forward(self, seeds, words, word_edges):
        # seeds: (B * C, 1, D); words: (B * C, 4, D); word_edges: (B * C, 4, E)
        bias = None
        if self.edge_bias_scale != 0:
            bias = self.edge_bias_scale * torch.tanh(self.edge_projection(word_edges))
            bias = bias.permute(0, 2, 1).reshape(-1, 1, words.shape[1]).to(seeds.dtype)
        attended = self.attn(seeds, words, words, attn_mask=bias, need_weights=False)[0]
        x = self.h_norm(seeds + attended)
        return self.mab_norm(self.residual_proj(x) + self.ff(x))


class GraphSetDecoder(SetDecoder):
    def __init__(self, width, heads, dropout, edge_dim, edge_bias_scale):
        super().__init__(width, width, width, num_seeds=1, mab_attn_heads=heads, dropout=dropout)
        original = self.mab
        self.mab = GraphCrossAttentionBlock(width, heads, dropout, edge_dim, edge_bias_scale)
        self.mab.load_state_dict(original.state_dict(), strict=False)

    def forward(self, x, edges):
        # x: (B * C, 4, D); edges: (B * C, 4, 4, E)
        words = self.ff1(x)
        seeds = self.seed_vectors.unsqueeze(0).expand(x.shape[0], -1, -1)
        # Diagonal edges are zero, so divide by the three other words.
        word_edges = edges.sum(dim=1) / (edges.shape[1] - 1)  # (B * C, 4, E)
        return self.mab(seeds, words, word_edges)


class GraphSetTransformer(SetTransformer):
    """Existing LTR scorer with edge-biased global and candidate attention."""
    def __init__(self, *, edge_dim=3, edge_bias_scale=1.0, **model_config):
        super().__init__(**model_config)
        self.edge_dim = edge_dim
        original = self.full_encoder
        first = original.mabs[0]
        self.full_encoder = GraphEncoder(
            first.attn.embed_dim, len(original.mabs), first.attn.num_heads,
            first.attn.dropout, edge_dim, edge_bias_scale,
        )
        # Preserve the original initialization; scale=0 is the same model.
        self.full_encoder.load_state_dict(original.state_dict(), strict=False)
        original = self.subset_encoder
        first = original.mabs[0]
        self.subset_encoder = GraphEncoder(
            first.attn.embed_dim, len(original.mabs), first.attn.num_heads,
            first.attn.dropout, edge_dim, edge_bias_scale,
        )
        self.subset_encoder.load_state_dict(original.state_dict(), strict=False)
        original = self.subset_decoder
        self.subset_decoder = GraphSetDecoder(
            original.mab.attn.embed_dim, original.mab.attn.num_heads,
            original.mab.attn.dropout, edge_dim, edge_bias_scale,
        )
        self.subset_decoder.load_state_dict(original.state_dict(), strict=False)

    def forward(self, x, group_idx=None, edge_features=None):
        B, N, _ = x.shape
        if edge_features is None or edge_features.shape != (B, N, N, self.edge_dim):
            raise ValueError("Expected edge_features with shape (B, N, N, edge_dim)")
        x = self.proj(self.input_norm(x))  # (B, N, D)
        x = self.full_encoder(x, edge_features)  # (B, N, D)
        idx = self.grouper.idx if group_idx is None else group_idx.to(x.device)
        x = self.grouper(x, group_idx=idx)  # (B, C, 4, D)
        B, C, group_size, D = x.shape
        # Select the same four indices on both edge axes for every candidate.
        candidate_edges = edge_features[:, idx[:, :, None], idx[:, None, :], :]
        candidate_edges = candidate_edges.reshape(B * C, group_size, group_size, self.edge_dim)
        x = x.reshape(B * C, group_size, D)
        x = self.subset_encoder(x, candidate_edges)  # (B * C, 4, D)
        x = self.subset_decoder(x, candidate_edges).squeeze(1)  # (B * C, D)
        return self.scorer(x.reshape(B, C, D))  # (B, C, 1)


def score_batch(model, x, batch, group_idx=None):
    kwargs = {}
    if isinstance(model, GraphSetTransformer):
        kwargs["edge_features"] = batch["edge_features"].to(x.device)
    return model(x, group_idx=group_idx, **kwargs)


def load_graph_harness(weights_path, device="auto", beam_width=20, state_batch_size=4,
                       cache_dir="data/feature_cache/graph"):
    """Reload this graph configuration; run imports and graph model definitions first."""
    checkpoint = torch.load(weights_path, map_location="cpu", weights_only=True)
    if checkpoint["model_class"] != "GraphSetTransformer" or "feature_metadata" not in checkpoint:
        raise ValueError("Expected a text/spelling graph checkpoint with feature metadata")
    resolved = ("cuda" if torch.cuda.is_available() else "cpu") if device == "auto" else device
    options = checkpoint["feature_metadata"]["options"]
    embedder = SentenceTransformer(checkpoint["embedding_model_name"],
                                  revision=options["revision"], device=str(resolved))
    nodes, edges, metadata = build_graph_features(
        embedder, checkpoint["embedding_model_name"], cache_dir=cache_dir, **options
    )
    if metadata != checkpoint["feature_metadata"] or list(edges.feature_names) != checkpoint["edge_feature_names"]:
        raise ValueError("Feature definitions or lexical resources differ from the checkpoint")
    config = checkpoint["model_config"]
    if config["in_dim"] != nodes.dim or config["edge_dim"] != edges.dim:
        raise ValueError("Checkpoint feature dimensions do not match")
    model = GraphSetTransformer(group_idx=torch.tensor([[0, 1, 2, 3]]), **config)
    model.load_state_dict(checkpoint["model_state_dict"])
    harness = SearchHarness.from_model(
        model, embedder, input_dim=nodes.dim, device=resolved,
        node_feature_builder=nodes, edge_feature_builder=edges.encode,
        beam_width=beam_width, state_batch_size=state_batch_size,
    )
    harness.checkpoint = checkpoint
    harness.weights_path = Path(weights_path)
    return harness


In [ ]:
from pytorchltr.loss import LambdaNDCGLoss2
from pytorchltr.evaluation import ndcg

# Constants

In [ ]:
DATASET_PATH = Path("data/connections.csv")
RELEVANCE_MODE = "binary"  # "binary" (exact match) or "graded" (maximum overlap)

M = 1820

# Helper Functions

In [ ]:
def extract_features(word):
    return pd.DataFrame({word:{
        'length': len(word),
        'num_vowels': sum(c in 'aeiou' for c in word.lower()),
        'num_consonants': sum(c.isalpha() and c not in 'aeiou' for c in word.lower()),
        'ends_with_ing': int(word.lower().endswith('ing')),
        'ends_with_ed': int(word.lower().endswith('ed')),
        'is_palindrome': int(word.lower() == word[::-1].lower())
    }})

def extract_structural_features(words):
    """
    Extract basic structural features for the given words.
    """
    return torch.tensor(pd.concat([extract_features(w) for w in words], axis=1).T.values) 

In [ ]:

def construct_groups_and_labels(num_sets_remaining=4, relevance_mode="binary"):
    """
    Produce candidate indices and binary or graded relevance labels.

    Labels assume the state's true groups are laid out consecutively in word order.
    """

    if num_sets_remaining < 1 or num_sets_remaining > 4:
        raise ValueError("num_sets_remaining must be between 1 and 4")
    if relevance_mode not in {"binary", "graded"}:
        raise ValueError("relevance_mode must be 'binary' or 'graded'")

    n_words = num_sets_remaining * 4
    groups = [set(range(i * 4, (i + 1) * 4)) for i in range(num_sets_remaining)]

    quads = list(itertools.combinations(range(n_words), 4))

    if relevance_mode == "binary":
        rel_scores = [int(set(q) in groups) for q in quads]
    else:
        rel_scores = [max(len(set(q).intersection(g)) for g in groups) for q in quads]

    return torch.tensor(quads), torch.tensor(rel_scores)

In [ ]:
from nyt_connections_solver.data_loader import collate_keep_wordlists


# Create Dataset

Rerun from the top after changing `DATASET_PATH`; existing outputs may reflect an earlier dataset.

In [ ]:
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)

In [ ]:
print(f"Cuda available: {torch.cuda.is_available()}")

Cuda available: True


In [ ]:
# Multilingual frozen encoder (100+ languages); D is inferred below.
EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-0.6B"
# Use symmetric, unprompted word embeddings in both training and the harness.
EMBEDDING_REVISION = "97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3"
word_embedder = SentenceTransformer(EMBEDDING_MODEL, revision=EMBEDDING_REVISION)
word_embedder.requires_grad_(False)
word_embedder.eval()

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'Qwen3Model'})
  (1): Pooling({'embedding_dimension': 1024, 'pooling_mode': 'lasttoken', 'include_prompt': True})
  (2): Normalize({})
)

In [ ]:
test_emb = word_embedder.encode("Hello")

In [ ]:
D = len(test_emb)
D

1024

In [ ]:
# ft = load_facebook_vectors('pretrained/cc.en.300.bin')

In [ ]:
class FastTextEmbedder:
    """
    Wrap gensim embeddings for simpler conversion to tensors and to fit with sentence-transformers api for easy swapping
    """
    def __init__(self, ft_model):
        self.ft = ft_model
        self.dim = ft_model.vector_size

    def encode(self, word_list, convert_to_tensor=True):
        vectors = []
        for word in word_list:
            vectors.append(self.ft[word])

        vectors = np.stack(vectors)

        if convert_to_tensor:
            return torch.tensor(vectors, dtype=torch.float)
        else:
            return vectors

In [ ]:
# word_embedder = FastTextEmbedder(ft)

In [ ]:
connections_df = pd.read_csv(DATASET_PATH, index_col=0)
# connections_df.drop(columns='category', inplace=True)

# fix issues with incorrect format... (not worth going in and adding a mechanism to correct...)
connections_df.loc[1298, "connections"] = "['line', 'plane', 'point', 'solid']"
connections_df.loc[1892, "connections"] = "['abyss', 'fly', 'matrix', 'thing']"

# remove april fools samples since they include emojis or other potentially noisy samples
connections_df = connections_df[~connections_df['date'].str.contains("04-01")]
connections_df.reset_index(drop=True, inplace=True)

connections_df['connections'] = connections_df['connections'].apply(ast.literal_eval)
connections_df.head()

,date,category,connections
0,2023-06-12,palindromes,"[kayak, level, mom, race car]"
1,2023-06-12,keyboard keys,"[option, return, shift, tab]"
2,2023-06-12,nba teams,"[bucks, heat, jazz, nets]"
3,2023-06-12,wet weather,"[hail, rain, sleet, snow]"
4,2023-06-13,letter homophones,"[are, queue, sea, why]"


In [ ]:
all_words = list(itertools.chain.from_iterable(
    connections_df
    .groupby('date')['connections']
    .agg(lambda lists: list(itertools.chain.from_iterable(lists)))
    .values
))
all_words[:10]

['kayak',
 'level',
 'mom',
 'race car',
 'option',
 'return',
 'shift',
 'tab',
 'bucks',
 'heat']

In [ ]:
# [w for w in all_words if ('_' in w) or ('-' in w)]

In [ ]:
# Character n-grams are hashed by TextSpellingFeatures; no fitted vocabulary is needed.
vectorizer = None


In [ ]:
from nyt_connections_solver.graph_features import build_graph_features
from nyt_connections_solver.node_features import TextSpellingFeatures

FEATURE_CACHE_DIR = Path("data/feature_cache/graph")
REFRESH_FEATURE_CACHE = False
base_feature_builder, graph_edge_builder, feature_metadata = build_graph_features(
    word_embedder, EMBEDDING_MODEL, cache_dir=FEATURE_CACHE_DIR,
    refresh=REFRESH_FEATURE_CACHE, revision=EMBEDDING_REVISION,
)
graph_edge_providers = graph_edge_builder.providers
print("Node dimensions:", base_feature_builder.dim)
print("Edge channels:", graph_edge_builder.feature_names)


Node dimensions: 1165
Edge channels: ('semantic_cosine', 'sense_jaccard', 'rare_concept_jaccard', 'longest_common_substring', 'common_prefix', 'common_suffix', 'prefix_semantic_similarity', 'suffix_semantic_similarity', 'shared_preceding_word', 'shared_following_word', 'exact_anagram', 'character_multiset_overlap', 'both_palindromes', 'both_initial_double', 'both_consecutive_double_pairs')


In [ ]:
from nyt_connections_solver.data_loader import GraphConnectionsDataset, make_state_loaders


In [ ]:
idx, y = construct_groups_and_labels(relevance_mode=RELEVANCE_MODE)

In [ ]:
idx[:10]

tensor([[ 0,  1,  2,  3],
        [ 0,  1,  2,  4],
        [ 0,  1,  2,  5],
        [ 0,  1,  2,  6],
        [ 0,  1,  2,  7],
        [ 0,  1,  2,  8],
        [ 0,  1,  2,  9],
        [ 0,  1,  2, 10],
        [ 0,  1,  2, 11],
        [ 0,  1,  2, 12]])

In [ ]:
idx.unique()

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15])

In [ ]:
y[:10]

tensor([1, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [ ]:
y.unique()

tensor([0, 1])

In [ ]:
# split data
dates = connections_df["date"].unique()
train_idx = random.sample(range(len(dates)), k=int(len(dates) * .8))
train_dates = dates[train_idx]
test_dates = dates[~np.isin(range(len(dates)), train_idx)]

test_dates = np.random.permutation(test_dates)
split_idx = len(test_dates) // 2
val_dates = test_dates[split_idx:]
test_dates = test_dates[:split_idx]

print(f"Total Length: {len(dates)}")
print(f"Train length: {len(train_dates)}")
print(f"Val length: {len(val_dates)}")
print(f"Test length: {len(test_dates)}")

Total Length: 1210
Train length: 968
Val length: 121
Test length: 121


In [ ]:
BATCH_SIZE = 16

# These prepared datasets/loaders are reused below; no duplicate preprocessing.
state_datasets = {}
state_loaders = {}
for split, dates_for_split in (("train", train_dates), ("val", val_dates), ("test", test_dates)):
    state_datasets[split], state_loaders[split] = make_state_loaders(
        connections_df.loc[connections_df["date"].isin(dates_for_split)],
        word_embedder, vectorizer, batch_size=BATCH_SIZE, shuffle=(split == "train"),
        feature_builder=base_feature_builder, edge_providers=graph_edge_providers,
        cache=graph_edge_builder.cache,
    )
train_state_datasets, val_state_datasets, test_state_datasets = (state_datasets[s] for s in ("train", "val", "test"))
train_state_loaders, val_state_loaders, test_state_loaders = (state_loaders[s] for s in ("train", "val", "test"))
train, val, test = (state_datasets[s][4] for s in ("train", "val", "test"))
train_loader, val_loader, test_loader = (state_loaders[s][4] for s in ("train", "val", "test"))


In [ ]:
idx[0]

tensor([0, 1, 2, 3])

In [ ]:
idx.shape

torch.Size([1820, 4])

In [ ]:
batch = next(iter(val_loader))
print("Node features:", batch["x"].shape)
print("Edge features:", batch["edge_features"].shape)
print("Words:", batch["word_list"][0])


Node features: torch.Size([16, 16, 1165])
Edge features: torch.Size([16, 16, 16, 15])
Words: ['cray', 'jelly', 'silver', 'star', 'can', 'head', 'john', 'throne', 'calf', 'cub', 'joey', 'kid', 'ankle', 'knee', 'shin', 'thigh']


In [ ]:
idx.shape

torch.Size([1820, 4])

# Model Architectures

## Model classes
The notebook uses shared `Baseline`, `AttentionModel`, and `SetTransformer`
architectures with an ordinary learned projection of the concatenated text and
spelling node features. The graph model additionally biases attention using the
named pairwise edge channels.


# Train Model

In [ ]:
def ranking_metrics(scores, relevance, n, hit_k=4, ndcg_k=4):
    """Return per-state ranking metrics where any positive subset is a success."""
    logits = scores.squeeze(-1)
    positive = relevance == relevance.max(dim=1, keepdim=True).values
    if logits.shape != positive.shape:
        raise ValueError(f"Scores {logits.shape} and relevance {positive.shape} must match")
    if not positive.any(dim=1).all():
        raise ValueError("Every state must contain at least one correct subset")

    order = logits.argsort(dim=1, descending=True)
    ranked_positive = positive.gather(1, order)
    first_rank = ranked_positive.float().argmax(dim=1) + 1
    ranks = torch.arange(1, logits.size(1) + 1, device=logits.device).float()
    precision_at_rank = ranked_positive.float().cumsum(dim=1) / ranks.unsqueeze(0)
    average_precision = (
        precision_at_rank * ranked_positive.float()
    ).sum(dim=1) / positive.sum(dim=1)

    hit_cutoff = min(hit_k, logits.size(1))
    ndcg_cutoff = min(ndcg_k, logits.size(1))
    return {
        "hit1": ranked_positive[:, 0].float(),
        "mrr": first_rank.float().reciprocal(),
        "hit4": ranked_positive[:, :hit_cutoff].any(dim=1).float(),
        "map": average_precision,
        "ndcg4": ndcg(scores, relevance, n, k=ndcg_cutoff),
    }


In [ ]:
def evaluate(model, loader, loss_fn, y, n_scalar, device, group_idx=None):
    """Evaluate loss and decision-focused ranking metrics over a loader."""
    model.eval()
    totals = {name: 0.0 for name in ("loss", "hit1", "mrr", "hit4", "map", "ndcg4")}
    total_examples = 0

    with torch.inference_mode():
        for batch in loader:
            x = batch["x"].to(device)
            batch_size = len(x)
            scores = score_batch(model, x, batch, group_idx=group_idx)
            relevance = y.unsqueeze(0).expand(batch_size, len(y))
            lengths = n_scalar.expand(batch_size)
            totals["loss"] += loss_fn(scores, relevance, lengths).mean().item() * batch_size
            batch_metrics = ranking_metrics(scores, relevance, lengths)
            for name, values in batch_metrics.items():
                totals[name] += values.sum().item()
            total_examples += batch_size

    if total_examples == 0:
        raise ValueError("Cannot evaluate an empty loader")
    return {name: value / total_examples for name, value in totals.items()}


In [ ]:
from tensorboardX import SummaryWriter
from datetime import datetime


def train(model, y, num_epochs, model_name, device, lr=1e-4):
    """Legacy full-state trainer, with decision-focused validation metrics."""
    current_time = datetime.now().strftime("%Y-%m-%d_%H-%M")
    writer = SummaryWriter(log_dir=f"runs/{model_name}/connections_experiment_LTR-{current_time}")
    scaler = torch.amp.GradScaler(
        torch.device(device).type, enabled=torch.device(device).type == "cuda"
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", patience=5)
    y = y.to(device)
    n_scalar = torch.tensor(M, device=device, dtype=torch.long)
    loss_fn = LambdaNDCGLoss2().to(device)

    for epoch in range(num_epochs):
        model.train()
        totals = {name: 0.0 for name in ("loss", "hit1", "mrr", "hit4", "map", "ndcg4")}
        total_examples = 0
        for batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            x = batch["x"].to(device)
            batch_size = len(x)
            relevance = y.unsqueeze(0).expand(batch_size, len(y))
            lengths = n_scalar.expand(batch_size)
            scores = model(x)
            batch_loss = loss_fn(scores, relevance, lengths).mean()
            scaler.scale(batch_loss).backward()
            scaler.step(optimizer)
            scaler.update()
            with torch.no_grad():
                batch_metrics = ranking_metrics(scores, relevance, lengths)
                totals["loss"] += batch_loss.item() * batch_size
                for name, values in batch_metrics.items():
                    totals[name] += values.sum().item()
                total_examples += batch_size

        train_metrics = {name: value / total_examples for name, value in totals.items()}
        val_metrics = evaluate(model, val_loader, loss_fn, y, n_scalar, device)
        scheduler.step(val_metrics["loss"])

        if epoch % 5 == 0 or epoch == num_epochs - 1:
            for name, value in train_metrics.items():
                writer.add_scalar(f"{name}/train", value, epoch)
            for name, value in val_metrics.items():
                writer.add_scalar(f"{name}/val", value, epoch)
            writer.add_scalar("lr", optimizer.param_groups[0]["lr"], epoch)
            print(
                f"Epoch {epoch} | Train Loss: {train_metrics['loss']:.4f} | "
                f"Train Hit@1: {train_metrics['hit1']:.4f} | Train MRR: {train_metrics['mrr']:.4f} | "
                f"Val Loss: {val_metrics['loss']:.4f} | Val Hit@1: {val_metrics['hit1']:.4f} | "
                f"Val MRR: {val_metrics['mrr']:.4f} | Val Hit@4: {val_metrics['hit4']:.4f}"
            )
    writer.close()
    return model.to("cpu")


def evaluate_full_state(model, loader, y, device, k=4):
    y = y.to(device)
    n_scalar = torch.tensor(len(y), device=device, dtype=torch.long)
    loss_fn = LambdaNDCGLoss2().to(device)
    return evaluate(model, loader, loss_fn, y, n_scalar, device)


In [ ]:
def make_state_artifacts(device=None, relevance_mode="binary"):
    artifacts = {}
    for size in (2, 3, 4):
        group_idx, relevance = construct_groups_and_labels(size, relevance_mode=relevance_mode)
        if device is not None:
            group_idx = group_idx.to(device)
            relevance = relevance.to(device)

        artifacts[size] = {
            "idx": group_idx,
            "y": relevance,
            "n": torch.tensor(len(relevance), device=device, dtype=torch.long) if device is not None else torch.tensor(len(relevance), dtype=torch.long),
            "k": size,
        }
    return artifacts


def _next_batch(iterators, loaders, size):
    try:
        return next(iterators[size])
    except StopIteration:
        iterators[size] = iter(loaders[size])
        return next(iterators[size])


def evaluate_by_state_size(model, loaders, artifacts, loss_fn, device):
    metrics = {}
    weighted = {name: 0.0 for name in ("loss", "hit1", "mrr", "hit4", "map", "ndcg4")}
    total_examples = 0
    for size, loader in loaders.items():
        artifact = artifacts[size]
        values = evaluate(
            model, loader, loss_fn, artifact["y"].to(device), artifact["n"].to(device),
            device, group_idx=artifact["idx"].to(device),
        )
        examples = len(loader.dataset)
        metrics[size] = {**values, "examples": examples}
        total_examples += examples
        for name in weighted:
            weighted[name] += values[name] * examples
    if not total_examples:
        raise ValueError("Cannot evaluate empty state loaders")
    metrics["weighted"] = {
        **{name: value / total_examples for name, value in weighted.items()},
        "examples": total_examples,
    }
    return metrics

def format_state_metrics(prefix, metrics):
    parts = [
        f"{prefix} Loss: {metrics['weighted']['loss']:.4f}",
        f"{prefix} Hit@1: {metrics['weighted']['hit1']:.4f}",
        f"{prefix} MRR: {metrics['weighted']['mrr']:.4f}",
        f"{prefix} Hit@4: {metrics['weighted']['hit4']:.4f}",
        f"{prefix} MAP: {metrics['weighted']['map']:.4f}",
        f"{prefix} NDCG@4: {metrics['weighted']['ndcg4']:.4f}",
    ]
    for size in (2, 3, 4):
        if size in metrics:
            parts.append(f"{prefix} s{size} Hit@1: {metrics[size]['hit1']:.4f}")
            parts.append(f"{prefix} s{size} MRR: {metrics[size]['mrr']:.4f}")
    return " | ".join(parts)

def train_curriculum_mixed(
    model, train_loaders, val_loaders, artifacts, phases, model_name, device,
    lr=5e-5, weight_decay=1e-5, steps_per_epoch=None, selection_size=None,
):
    """Legacy curriculum trainer; checkpoint selection uses validation Hit@1."""
    current_time = datetime.now().strftime("%Y-%m-%d_%H-%M")
    writer = SummaryWriter(log_dir=f"runs/{model_name}/connections_curriculum_LTR-{current_time}")
    model = model.to(device)
    loss_fn = LambdaNDCGLoss2().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", patience=5)
    scaler = torch.amp.GradScaler(
        torch.device(device).type, enabled=torch.device(device).type == "cuda"
    )
    artifacts = {
        size: {key: value.to(device) if torch.is_tensor(value) else value for key, value in item.items()}
        for size, item in artifacts.items()
    }
    iterators = {size: iter(loader) for size, loader in train_loaders.items()}
    best_val_hit1 = -float("inf")
    best_val_mrr = -float("inf")
    best_state_dict = None
    global_epoch = 0

    for phase in phases:
        active_sizes = phase["sizes"]
        weights = phase.get("weights", [1 / len(active_sizes)] * len(active_sizes))
        phase_steps = steps_per_epoch or max(len(train_loaders[size]) for size in active_sizes)
        for phase_epoch in range(phase["epochs"]):
            model.train()
            train_totals = {name: 0.0 for name in ("loss", "hit1", "mrr", "hit4", "map", "ndcg4")}
            for _ in range(phase_steps):
                size = random.choices(active_sizes, weights=weights, k=1)[0]
                artifact = artifacts[size]
                batch = _next_batch(iterators, train_loaders, size)
                optimizer.zero_grad(set_to_none=True)
                x = batch["x"].to(device)
                relevance = artifact["y"].unsqueeze(0).expand(len(x), -1)
                lengths = artifact["n"].expand(len(x))
                scores = score_batch(model, x, batch, group_idx=artifact["idx"])
                batch_loss = loss_fn(scores, relevance, lengths).mean()
                scaler.scale(batch_loss).backward()
                scaler.step(optimizer)
                scaler.update()
                with torch.no_grad():
                    batch_metrics = ranking_metrics(scores, relevance, lengths)
                    train_totals["loss"] += batch_loss.item()
                    for name, values in batch_metrics.items():
                        train_totals[name] += values.mean().item()
            train_metrics = {name: value / phase_steps for name, value in train_totals.items()}
            val_metrics = evaluate_by_state_size(
                model, {size: val_loaders[size] for size in active_sizes}, artifacts, loss_fn, device
            )
            selection_key = "weighted" if selection_size is None else selection_size
            if selection_key not in val_metrics:
                raise ValueError(f"Selection state size {selection_size} is not active")
            selection_metrics = val_metrics[selection_key]
            scheduler.step(selection_metrics["loss"])
            if selection_metrics["hit1"] > best_val_hit1:
                best_val_hit1 = selection_metrics["hit1"]
                best_state_dict = {
                    key: value.detach().cpu().clone() for key, value in model.state_dict().items()
                }
            for name, value in train_metrics.items():
                writer.add_scalar(f"{name}/train", value, global_epoch)
            for name in ("loss", "hit1", "mrr", "hit4", "map", "ndcg4"):
                writer.add_scalar(f"{name}/val_weighted", val_metrics["weighted"][name], global_epoch)
            if phase_epoch % 5 == 0 or phase_epoch == phase["epochs"] - 1:
                print(
                    f"Phase {phase['name']} Epoch {phase_epoch} | "
                    f"Train Loss: {train_metrics['loss']:.4f} | "
                    f"Train Hit@1: {train_metrics['hit1']:.4f} | "
                    f"{format_state_metrics('Val', val_metrics)}"
                )
            global_epoch += 1
    writer.close()
    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)
    return model.to("cpu")

def evaluate_state_ranking(model, loaders, artifacts, device):
    """Evaluate decision-focused rankings without running the search harness."""
    model.eval()
    metrics = {}
    with torch.inference_mode():
        for size, loader in loaders.items():
            artifact = artifacts[size]
            totals = {name: 0.0 for name in ("hit1", "mrr", "hit4", "map", "ndcg4")}
            count = 0
            group_idx = artifact["idx"].to(device)
            y = artifact["y"].to(device)
            n = artifact["n"].to(device)
            for batch in loader:
                x = batch["x"].to(device)
                scores = score_batch(model, x, batch, group_idx=group_idx)
                relevance = y.unsqueeze(0).expand(len(x), -1)
                lengths = n.expand(len(x))
                batch_metrics = ranking_metrics(scores, relevance, lengths)
                for name, values in batch_metrics.items():
                    totals[name] += values.sum().item()
                count += len(x)
            if not count:
                raise ValueError("Cannot validate an empty loader")
            metrics[size] = {name: value / count for name, value in totals.items()}
    return metrics

def state_loss(loss_mode, scores, relevance, n):
    """Per-sample state loss. `scores` is (B, C, 1) logits; `relevance` is (B, C).

    "ltr"         -> LambdaNDCG over the C candidates (rank order; the control).
    "subset_prob" -> -log(softmax mass on the correct subsets). Because softmax mass sums to
                     1, this explicitly pushes probability OFF every incorrect subset, including
                     the top near-misses that LambdaNDCG weights softly via |dNDCG|.
    """
    if loss_mode == "ltr":
        return LambdaNDCGLoss2()(scores, relevance, n).mean()
    if loss_mode == "subset_prob":
        logits = scores.squeeze(-1)
        pos = relevance == relevance.max(dim=1, keepdim=True).values
        log_probs = torch.log_softmax(logits.float(), dim=1)
        masked = torch.where(pos, log_probs, torch.full_like(log_probs, float("-inf")))
        return -torch.logsumexp(masked, dim=1).mean()
    raise ValueError(f"Unknown LOSS_MODE: {loss_mode!r}")

def train_mixed_states(
    model, train_loaders, val_loaders, artifacts, device,
    lr, weight_decay, total_updates, eval_every,
    min_updates_before_stopping=700, early_stopping_patience=6, min_delta=1e-3,
    trial=None,
):
    """Train for a fixed update budget and select checkpoints by validation Hit@1."""
    if total_updates < 1 or eval_every < 1:
        raise ValueError("total_updates and eval_every must be positive")
    if min_updates_before_stopping < 0:
        raise ValueError("min_updates_before_stopping must be non-negative")
    if early_stopping_patience < 1 or min_delta < 0:
        raise ValueError("early_stopping_patience must be positive and min_delta non-negative")

    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scaler = torch.amp.GradScaler(torch.device(device).type, enabled=torch.device(device).type == "cuda")
    artifacts = {
        size: {key: value.to(device) if torch.is_tensor(value) else value for key, value in item.items()}
        for size, item in artifacts.items()
    }
    sizes = tuple(sorted(train_loaders))
    if set(val_loaders) != set(sizes):
        raise ValueError("Training and validation must use the same state sizes")
    iterators = {size: iter(loader) for size, loader in train_loaders.items()}

    best_val_hit1 = -float("inf")
    best_val_mrr = -float("inf")
    best_state_dict = None
    best_updates = 0
    patience_reference = -float("inf")
    stale_evals = 0
    evals_run = 0
    history = []

    def evaluate_once(step):
        nonlocal best_val_hit1, best_val_mrr, best_state_dict, best_updates
        nonlocal patience_reference, stale_evals, evals_run
        metrics = evaluate_state_ranking(model, val_loaders, artifacts, device)
        macro = {
            name: float(np.mean([metrics[size][name] for size in sizes]))
            for name in ("hit1", "mrr", "hit4", "map", "ndcg4")
        }
        val_hit1 = macro["hit1"]
        val_mrr = macro["mrr"]
        if not np.isfinite(val_hit1) or not np.isfinite(val_mrr):
            raise ValueError(f"Nonfinite validation Hit@1 at update {step}")
        evals_run += 1
        if (val_hit1 > best_val_hit1 or
                (val_hit1 == best_val_hit1 and val_mrr > best_val_mrr)):
            best_val_hit1, best_val_mrr, best_updates = val_hit1, val_mrr, step
            best_state_dict = {
                key: value.detach().cpu().clone() for key, value in model.state_dict().items()
            }
        if val_hit1 > patience_reference + min_delta:
            patience_reference, stale_evals = val_hit1, 0
        else:
            stale_evals += 1
        row = {
            "updates": step, "val_hit1": val_hit1, "val_mrr": val_mrr,
            "best_val_hit1": best_val_hit1, "best_val_mrr": best_val_mrr,
            "best_updates": best_updates, "lr": optimizer.param_groups[0]["lr"],
            **{f"val_{name}": value for name, value in macro.items()},
        }
        for size in sizes:
            row.update({f"val_{name}_s{size}": value for name, value in metrics[size].items()})
        history.append(row)
        if trial is not None:
            trial.report(val_hit1, step)
            trial.set_user_attr("evals_run", evals_run)
            trial.set_user_attr("updates_run", step)
            trial.set_user_attr("best_updates", best_updates)
            trial.set_user_attr("best_validation_hit1", best_val_hit1)
            trial.set_user_attr("best_validation_mrr", best_val_mrr)
        by_size = " | ".join(
            f"s{size} Hit@1: {metrics[size]['hit1']:.4f}, MRR: {metrics[size]['mrr']:.4f}"
            for size in sizes
        )
        print(
            f"Update {step}/{total_updates} | Val macro Hit@1: {val_hit1:.4f} | "
            f"MRR: {macro['mrr']:.4f} | Hit@4: {macro['hit4']:.4f} | "
            f"Best Hit@1: {best_val_hit1:.4f} @ {best_updates} | {by_size}"
        )

    model.train()
    pending_sizes = []
    step = 0
    while step < total_updates:
        if not pending_sizes:
            pending_sizes = list(sizes)
            random.shuffle(pending_sizes)
        size = pending_sizes.pop()
        batch = _next_batch(iterators, train_loaders, size)
        artifact = artifacts[size]
        x = batch["x"].to(device)
        relevance = artifact["y"].unsqueeze(0).expand(len(x), -1)
        lengths = artifact["n"].expand(len(x))
        optimizer.zero_grad(set_to_none=True)
        scores = score_batch(model, x, batch, group_idx=artifact["idx"])
        batch_loss = state_loss(LOSS_MODE, scores, relevance, lengths)
        if not torch.isfinite(batch_loss):
            raise ValueError(f"Nonfinite training loss at update {step + 1}")
        scaler.scale(batch_loss).backward()
        scaler.step(optimizer)
        scaler.update()
        step += 1
        if step % eval_every == 0 or step == total_updates:
            evaluate_once(step)
            if trial is not None and trial.should_prune():
                raise optuna.TrialPruned()
            if step >= min_updates_before_stopping and stale_evals >= early_stopping_patience:
                print(
                    f"Early stopping at {step} updates; restoring update {best_updates} "
                    f"(Hit@1 {best_val_hit1:.4f})."
                )
                break

    if best_state_dict is None:
        raise ValueError("No validation checkpoint was recorded; increase the update budget")
    model.load_state_dict(best_state_dict)
    model.training_summary = {
        "best_validation_hit1": best_val_hit1,
        "best_validation_mrr": best_val_mrr,
        "best_updates": best_updates,
        "updates_ran": step,
        "evals_run": evals_run,
        "total_updates": total_updates,
        "eval_every": eval_every,
        "min_updates_before_stopping": min_updates_before_stopping,
        "early_stopping_patience": early_stopping_patience,
        "min_delta": min_delta,
        "lr": lr,
        "weight_decay": weight_decay,
        "history": history,
    }
    return model.to("cpu")


In [ ]:
RUN_BASELINES = False  # optional comparisons; not required for graph tuning
EPOCHS = 100
M = 1820
BASELINE_EPOCHS = 50
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
state_artifacts = make_state_artifacts(relevance_mode=RELEVANCE_MODE)
cpu_device = torch.device("cpu")
cpu_artifacts = make_state_artifacts(device=cpu_device, relevance_mode=RELEVANCE_MODE)
ranking_loss = LambdaNDCGLoss2().to(cpu_device)

# Training objective for this experimental copy: "subset_prob" maximizes softmax mass on
# the correct subsets (suppressing incorrect ones, incl. top near-misses); "ltr" is the
# LambdaNDCG control (connections-graph.ipynb).
LOSS_MODE = "subset_prob"


## Baseline Model

Baseline models train on the same uniformly sampled mixture of 2-, 3-, and 4-group states as the Set Transformer. Their architectures and learning rates remain fixed; no baseline hyperparameter search is run. 

In [ ]:
if RUN_BASELINES:
    model = Baseline(
        in_dim=base_feature_builder.dim,
        group_idx=idx,
        layers=[512, 512, 512]
    ).to(device)

    steps_per_epoch = len(train_state_loaders[4])
    model = train_mixed_states(
        model, train_state_loaders, val_state_loaders, state_artifacts,
        device=device, lr=1e-4, weight_decay=1e-5,
        total_updates=BASELINE_EPOCHS * steps_per_epoch, eval_every=steps_per_epoch,
        min_updates_before_stopping=0, early_stopping_patience=6, min_delta=1e-4,
    )
    baseline_val_metrics = evaluate_by_state_size(model, val_state_loaders, cpu_artifacts, ranking_loss, cpu_device)
    baseline_test_metrics = evaluate_by_state_size(model, test_state_loaders, cpu_artifacts, ranking_loss, cpu_device)
    print(format_state_metrics("Baseline Val", baseline_val_metrics))
    print(format_state_metrics("Baseline Test", baseline_test_metrics))

## Attention Model

This model uses the same multi-size data mixture and validation selection as the MLP baseline, with a fixed architecture and learning rate. 

In [ ]:
if RUN_BASELINES:

    model = AttentionModel(
        in_dim=base_feature_builder.dim,
        group_idx=idx,
        layers=[1024, 512, 256],
        attn_layers=5
    ).to(device)

    steps_per_epoch = len(train_state_loaders[4])
    model = train_mixed_states(
        model, train_state_loaders, val_state_loaders, state_artifacts,
        device=device, lr=5e-5, weight_decay=1e-5,
        total_updates=BASELINE_EPOCHS * steps_per_epoch, eval_every=steps_per_epoch,
        min_updates_before_stopping=0, early_stopping_patience=6, min_delta=1e-4,
    )
    attention_val_metrics = evaluate_by_state_size(model, val_state_loaders, cpu_artifacts, ranking_loss, cpu_device)
    attention_test_metrics = evaluate_by_state_size(model, test_state_loaders, cpu_artifacts, ranking_loss, cpu_device)
    print(format_state_metrics("Attention Val", attention_val_metrics))
    print(format_state_metrics("Attention Test", attention_test_metrics))

## Graph Set Transformer

Optuna selects by macro validation Hit@1 across 2/3/4 remaining sets, with MRR
breaking ties. Architecture is fixed while dropout, learning rate, and AdamW
weight decay are tuned. Trials evaluate all validation states every 100 updates,
with early stopping and pruning available after 700 updates. The best checkpoint
is restored; final ranking metrics are reported after selection.

Set `RUN_FINAL_SEARCH = True` to additionally evaluate the selected model in the
feedback game environment (four mistakes, retries, and one-away pruning).
Search stays outside training and Optuna. Boards are reproducibly shuffled;
transcripts and timings are saved in `test_search_metrics["games"]`.


In [ ]:
OPTUNA_TRIALS = 24
TUNING_SEED = 0
RUN_FINAL_SEARCH = False
FINAL_BEAM_WIDTH = 20
FINAL_MAX_MISTAKES = 4
FINAL_SEARCH_SEED = 0

# The architecture stays fixed while optimizer and regularization settings are tuned.
LR_VALUES = [1e-4, 2e-4, 3e-4, 5e-4]
WEIGHT_DECAY_VALUES = [0.0, 1e-6, 1e-5, 1e-4]
TUNING_TOTAL_UPDATES = 1500
TUNING_EVAL_EVERY = 100
TUNING_MIN_UPDATES_BEFORE_STOP = 700
TUNING_PRUNER_WARMUP_UPDATES = 700     # Optuna cannot prune before this update
TUNING_EARLY_STOP_PATIENCE = 6       # evaluations without Hit@1 improvement after warm-up
TUNING_MIN_DELTA = 1e-3
TUNING_VAL_MAX_STATES_PER_SIZE = None  # use every validation state
DROPOUT_VALUES = [0.0, 0.05, 0.1, 0.2]

TUNING_REFERENCE_CONFIG = {
    "edge_dim": graph_edge_builder.dim,
    "edge_bias_scale": 0.5,
    "in_dim": base_feature_builder.dim,
    "lin_layers": 2,
    "lin_width": 512,
    "attn_layers": 1,
    "attn_width": 256,
    "attn_heads": 8,
}
# Architecture remains fixed while optimizer settings are compared.

from torch.utils.data import Subset

def tuning_validation_loaders(datasets, max_states, batch_size):
    if max_states is not None and max_states < 1:
        raise ValueError("max_states must be positive or None")
    loaders = {}
    for size, dataset in datasets.items():
        count = len(dataset) if max_states is None else min(len(dataset), max_states)
        indices = np.linspace(0, len(dataset) - 1, count, dtype=int).tolist()
        loaders[size] = DataLoader(Subset(dataset, indices), batch_size=batch_size,
                                  shuffle=False, collate_fn=collate_keep_wordlists)
    return loaders

def tuning_training_loaders(datasets, batch_size, seed):
    """Build fresh loaders so every trial starts from the same sampler state."""
    return {
        size: DataLoader(
            dataset, batch_size=batch_size, shuffle=True,
            collate_fn=collate_keep_wordlists,
            generator=torch.Generator().manual_seed(seed + size),
        )
        for size, dataset in datasets.items()
    }

tuning_val_loaders = tuning_validation_loaders(
    val_state_datasets, TUNING_VAL_MAX_STATES_PER_SIZE, BATCH_SIZE
)
retained_trial = {"score": -float("inf")}

def objective(trial):
    # Reset every RNG and rebuild the shuffled loaders below. This makes repeated
    # configurations matched runs instead of continuing sampler state across trials.
    random.seed(TUNING_SEED)
    np.random.seed(TUNING_SEED)
    torch.manual_seed(TUNING_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(TUNING_SEED)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    trial.set_user_attr("seed", TUNING_SEED)
    trial_train_loaders = tuning_training_loaders(
        train_state_datasets, BATCH_SIZE, TUNING_SEED
    )

    dropout = trial.suggest_categorical("dropout", DROPOUT_VALUES)
    lr = trial.suggest_categorical("lr", LR_VALUES)
    weight_decay = trial.suggest_categorical("weight_decay", WEIGHT_DECAY_VALUES)
    model_config = dict(TUNING_REFERENCE_CONFIG, dropout=dropout)
    trial_model = GraphSetTransformer(
        group_idx=state_artifacts[4]["idx"], **model_config
    ).to(device)
    try:
        trial_model = train_mixed_states(
            trial_model, trial_train_loaders, tuning_val_loaders, state_artifacts,
            device=device, lr=lr, weight_decay=weight_decay,
            total_updates=TUNING_TOTAL_UPDATES, eval_every=TUNING_EVAL_EVERY,
            min_updates_before_stopping=TUNING_MIN_UPDATES_BEFORE_STOP,
            early_stopping_patience=TUNING_EARLY_STOP_PATIENCE, min_delta=TUNING_MIN_DELTA,
            trial=trial,
        )
        summary = trial_model.training_summary
        hit1 = summary["best_validation_hit1"]
        mrr = summary["best_validation_mrr"]
        score = hit1 + 1e-6 * mrr  # MRR only breaks exact Hit@1 ties
        trial.set_user_attr("best_validation_hit1", hit1)
        trial.set_user_attr("best_validation_mrr", mrr)
        trial.set_user_attr("training_summary", summary)
        # Keep only the best completed model, not one large checkpoint per trial.
        if score > retained_trial["score"]:
            retained_trial.update(score=score, number=trial.number, config=model_config,
                                  state={k: v.detach().cpu().clone() for k, v in trial_model.state_dict().items()})
        return score
    finally:
        trial_model.to("cpu")
        if torch.device(device).type == "cuda":
            torch.cuda.empty_cache()


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=TUNING_SEED),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=TUNING_PRUNER_WARMUP_UPDATES,
        interval_steps=TUNING_EVAL_EVERY,
    ),
)
study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

best_trial = study.best_trial
assert retained_trial["number"] == best_trial.number
best_model_config = retained_trial["config"]
best_training_config = {
    "relevance_mode": RELEVANCE_MODE,
    "selection_metric": "macro_state_hit1",
    "seed": TUNING_SEED,
    "fresh_seeded_loaders_per_trial": True,
    "cudnn_deterministic": True,
    "lr": best_trial.params["lr"],
    "weight_decay": best_trial.params["weight_decay"],
    "dropout": best_trial.params["dropout"],
    "tie_break_metric": "macro_state_mrr",
    "total_updates": TUNING_TOTAL_UPDATES,
    "eval_every": TUNING_EVAL_EVERY,
    "min_updates_before_stopping": TUNING_MIN_UPDATES_BEFORE_STOP,
    "pruner_warmup_updates": TUNING_PRUNER_WARMUP_UPDATES,
    "early_stopping_patience": TUNING_EARLY_STOP_PATIENCE,
    "min_delta": TUNING_MIN_DELTA,
    "validation_max_states_per_size": TUNING_VAL_MAX_STATES_PER_SIZE,
    "training_summary": best_trial.user_attrs["training_summary"],
}
model = GraphSetTransformer(group_idx=state_artifacts[4]["idx"], **best_model_config)
model.load_state_dict(retained_trial["state"])
model = model.to("cpu").eval()
print("Best tuning validation Hit@1:", best_trial.user_attrs["best_validation_hit1"])
print("Best tuning validation MRR:", best_trial.user_attrs["best_validation_mrr"])
print("Best update:", best_trial.user_attrs["best_updates"])
optuna_results = study.trials_dataframe()
optuna_results

# Full ranking evaluation once after selection, on the configured device.
model = model.to(device).eval()
final_val_metrics = evaluate_state_ranking(model, val_state_loaders, state_artifacts, device)
final_test_metrics = evaluate_state_ranking(model, test_state_loaders, state_artifacts, device)
print("Full validation ranking metrics by remaining sets:", final_val_metrics)
print("Full test ranking metrics by remaining sets:", final_test_metrics)

# Search is optional and never part of the Optuna objective.
if RUN_FINAL_SEARCH:
    test_puzzles = load_puzzles(
        DATASET_PATH, pd.to_datetime(test_dates).strftime("%Y-%m-%d").tolist()
    )
    test_harness = SearchHarness.from_model(
        model, word_embedder, input_dim=base_feature_builder.dim,
        node_feature_builder=base_feature_builder, edge_feature_builder=graph_edge_builder.encode,
        beam_width=FINAL_BEAM_WIDTH, device=device,
    )
    feedback_harness = FeedbackSearchHarness.from_harness(test_harness)
    test_search_metrics = evaluate_feedback_games(
        feedback_harness, test_puzzles, max_mistakes=FINAL_MAX_MISTAKES,
        seed=FINAL_SEARCH_SEED,
    )
    print("Final test feedback metrics:", {
        key: value for key, value in test_search_metrics.items() if key != "games"
    })
    # test_search_metrics["games"] retains each public guess/feedback transcript.
    del feedback_harness, test_harness
model = model.to("cpu")


# Optional node-feature ablation
Compare text-only and text-plus-spelling nodes with matched seeds and the same
edge channels. Only validation ranking metrics are inspected here.


In [ ]:
def run_feature_ablation(epochs=20, seed=0):
    results = []
    for use_spelling in (False, True):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        nodes = TextSpellingFeatures(base_feature_builder.embeddings, use_spelling=use_spelling)
        loaders = {}
        for split, dates_for_split in (("train", train_dates), ("val", val_dates)):
            _, loaders[split] = make_state_loaders(
                connections_df.loc[connections_df["date"].isin(dates_for_split)],
                word_embedder, vectorizer, BATCH_SIZE, shuffle=(split == "train"),
                feature_builder=nodes, edge_providers=graph_edge_builder,
            )
        config = dict(best_model_config, in_dim=nodes.dim)
        candidate = GraphSetTransformer(group_idx=state_artifacts[4]["idx"], **config)
        steps_per_epoch = len(loaders["train"][4])
        candidate = train_mixed_states(
            candidate, loaders["train"], loaders["val"], state_artifacts,
            device=device, lr=best_training_config["lr"],
            weight_decay=best_training_config["weight_decay"],
            total_updates=epochs * steps_per_epoch, eval_every=steps_per_epoch,
            min_updates_before_stopping=0, early_stopping_patience=6, min_delta=1e-4,
        )
        metrics = evaluate_by_state_size(candidate, loaders["val"], cpu_artifacts, ranking_loss, cpu_device)
        results.append({"spelling": use_spelling, **metrics["weighted"]})
    return pd.DataFrame(results)


In [ ]:
RUN_FEATURE_ABLATION = False
if RUN_FEATURE_ABLATION:
    feature_ablation_results = run_feature_ablation()
    display(feature_ablation_results)


# Live inference
After saving, use `load_graph_harness(weights_path)` to reconstruct the frozen
node features and providers. Feature caching is shared with training; the first
unseen board can be slower than subsequent calls.

Wrap the loaded scorer with `FeedbackSearchHarness.from_harness(harness)` and
call `play_game(environment, policy)`. An environment implements `observe()`
returning `GameState` and `submit(guess)` returning `GuessFeedback`. Replace
`LocalConnectionsEnvironment` with a browser adapter to use the same policy live.
The policy sees only public words, accepted groups, feedback, and mistakes left.
A one-away result never reveals the correct triple.


# Save Graph Set Transformer Weights

Save the graph checkpoint and feature metadata. Reload with load_graph_harness above; the standard CLI does not load notebook-local graph classes.

In [ ]:
from datetime import datetime
weights_dir = Path("weights")
weights_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
weights_path = weights_dir / f"graph_set_transformer_optuna_text_spelling_{best_training_config['relevance_mode']}_{timestamp}.pt"

model_config = best_model_config.copy()
split_dates = {
    split_name: pd.to_datetime(split).strftime("%Y-%m-%d").tolist()
    for split_name, split in (("train", train_dates), ("val", val_dates), ("test", test_dates))
}
checkpoint = {
    "model_class": "GraphSetTransformer",
    "edge_feature_names": list(graph_edge_builder.feature_names),
    "edge_context": "original_16_words",
    "model_config": model_config,
    "training_config": best_training_config,
    "model_state_dict": {key: value.detach().cpu() for key, value in model.state_dict().items()},
    "embedding_model_name": EMBEDDING_MODEL,
    "feature_metadata": feature_metadata,
    "split_dates": split_dates,
    "dataset_path": str(DATASET_PATH.resolve()),
}
torch.save(checkpoint, weights_path)
print(f"Saved Graph Set Transformer checkpoint to {weights_path}")

Saved Graph Set Transformer checkpoint to weights/graph_set_transformer_optuna_text_spelling_binary_20261018_210543.pt
